# MNIST Fully Connected Autoencoder
A fully connected autoencoder trained on MNIST.

| Parameter | Value |
|-----------|-------|
| Input | 28×28 → 784 |
| Architecture | 784 → 256 → 128 → **32** → 128 → 256 → 784 |
| Latent dim | 32 |
| Loss | MSE |

## 1. Import Required Libraries

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 2. Load and Preprocess MNIST Dataset

In [ ]:
# Normalize to [0, 1] and keep as flat 784-dim vector for FC layers
transform = transforms.Compose([
    transforms.ToTensor(),          # scales to [0, 1]
    transforms.Lambda(lambda x: x.view(-1))  # 1x28x28 -> 784
])

train_dataset = torchvision.datasets.MNIST(root="./data", train=True,  download=True, transform=transform)
test_dataset  = torchvision.datasets.MNIST(root="./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True,  num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=256, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train samples : {len(train_dataset)}")
print(f"Test  samples : {len(test_dataset)}")
print(f"Input shape   : {train_dataset[0][0].shape}  (784-dim flattened)")

## 3. Define the Encoder

Compresses 784-dim input down to a 32-dim latent vector:

$$784 \xrightarrow{\text{FC+ReLU}} 256 \xrightarrow{\text{FC+ReLU}} 128 \xrightarrow{\text{FC+ReLU}} 32$$

In [ ]:
class Encoder(nn.Module):
    def __init__(self, input_dim: int = 784, latent_dim: int = 32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256), nn.ReLU(),
            nn.Linear(256, 128),       nn.ReLU(),
            nn.Linear(128, latent_dim),nn.ReLU(),
        )

    def forward(self, x):
        return self.net(x)

## 4. Define the Decoder

Reconstructs the 784-dim output from the 32-dim latent vector:

$$32 \xrightarrow{\text{FC+ReLU}} 128 \xrightarrow{\text{FC+ReLU}} 256 \xrightarrow{\text{FC+Sigmoid}} 784$$

The final **Sigmoid** maps outputs to $[0, 1]$, matching the normalized pixel range.

In [ ]:
class Decoder(nn.Module):
    def __init__(self, latent_dim: int = 32, output_dim: int = 784):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 128), nn.ReLU(),
            nn.Linear(128, 256),        nn.ReLU(),
            nn.Linear(256, output_dim), nn.Sigmoid(),  # outputs in [0,1]
        )

    def forward(self, z):
        return self.net(z)

## 5. Build the Autoencoder Model

In [ ]:
class Autoencoder(nn.Module):
    def __init__(self, input_dim: int = 784, latent_dim: int = 32):
        super().__init__()
        self.encoder = Encoder(input_dim, latent_dim)
        self.decoder = Decoder(latent_dim, input_dim)

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z)

    def encode(self, x):
        return self.encoder(x)


LATENT_DIM = 32
model = Autoencoder(input_dim=784, latent_dim=LATENT_DIM).to(device)
print(model)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTrainable parameters: {total_params:,}")

## 6. Train the Autoencoder

**MSE loss:**
$$\mathcal{L} = \frac{1}{n}\sum_{i=1}^{n}(x_i - \hat{x}_i)^2$$

Optimiser: **Adam** (lr = 1e-3), 20 epochs.

In [ ]:
NUM_EPOCHS = 20
LEARNING_RATE = 1e-3

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

train_losses = []

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    running_loss = 0.0
    for images, _ in train_loader:
        images = images.to(device)          # (B, 784)
        recon  = model(images)              # (B, 784)
        loss   = criterion(recon, images)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)

    epoch_loss = running_loss / len(train_dataset)
    train_losses.append(epoch_loss)
    print(f"Epoch [{epoch:2d}/{NUM_EPOCHS}]  MSE Loss: {epoch_loss:.6f}")

# Plot training loss
plt.figure(figsize=(8, 4))
plt.plot(range(1, NUM_EPOCHS + 1), train_losses, marker="o", linewidth=2)
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("Training Loss")
plt.grid(True)
plt.tight_layout()
plt.show()

## 7. Visualize Reconstructions

Side-by-side comparison of original vs reconstructed test images.

In [ ]:
model.eval()
NUM_DISPLAY = 10

# Grab one batch from the test loader
test_images, _ = next(iter(test_loader))
test_images = test_images.to(device)

with torch.no_grad():
    reconstructed = model(test_images).cpu()

test_images_cpu = test_images.cpu()

fig, axes = plt.subplots(2, NUM_DISPLAY, figsize=(NUM_DISPLAY * 1.4, 3))
for i in range(NUM_DISPLAY):
    # Original
    axes[0, i].imshow(test_images_cpu[i].view(28, 28), cmap="gray")
    axes[0, i].axis("off")
    if i == 0:
        axes[0, i].set_title("Original", fontsize=9)

    # Reconstructed
    axes[1, i].imshow(reconstructed[i].view(28, 28), cmap="gray")
    axes[1, i].axis("off")
    if i == 0:
        axes[1, i].set_title("Reconstructed", fontsize=9)

plt.suptitle("Original vs Reconstructed MNIST Digits", y=1.02)
plt.tight_layout()
plt.show()

## 8. Explore the Latent Space

Encode all test images into 32-dim latent vectors, reduce to 2D with **t-SNE**, and colour by digit class.